# RotorQuant Local Quantisation
**No Colab needed.** This notebook handles two local workflows:
1. **Path A — LoRA Conversion**: Converts raw `safetensors` LoRA adapters (from training) to GGUF. (⚠️ Not yet supported for Gemma 4 VLM architecture).
2. **Path B — Quantisation**: Re-quantises the existing merged `gemma4-legal-vlm` Ollama blob
(which already has the legal fine-tune baked in) from Q4_K_M → IQ4_XS so it can run under the
RotorQuant / AtomicBot inference stack.

### What this does
1. Locates the Ollama blob on disk (it is already a valid GGUF file)
2. Copies it with a readable name
3. Calls `llama-quantize.exe` with `--allow-requantize` to convert to IQ4_XS
4. Prints the env var line to paste into `.env` 

### Prerequisites
- `llama-quantize.exe` from the same build as your `llama-server.exe`
  (both live in `C:\Users\james\Desktop\llama-server-cuda\` by default)
- ~10 GB free disk space

In [ ]:
import os, shutil, subprocess, pathlib, getpass

# ── Where llama-quantize.exe lives (same folder as llama-server.exe) ──────────
LLAMA_QUANTIZE = os.environ.get(
    "LLAMA_QUANTIZE_PATH",
    r"C:\Users\james\Desktop\llama-server-cuda\llama-quantize.exe"
)

# ── Output directory (this is where the already-completed IQ4_XS files live) ──
OUT_DIR = pathlib.Path(r"C:\Users\james\Desktop\gemma4-legal-iq4xs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IQ4XS_PATH = OUT_DIR / "gemma4-legal-iq4xs.gguf"
IQ4XS_DIRECT_PATH = OUT_DIR / "gemma4-legal-iq4xs-direct.gguf"
Q4KM_PATH  = OUT_DIR / "gemma4-legal-merged-q4km.gguf"
F16_PATH   = OUT_DIR / "gemma4-legal-f16.gguf"

# ── Source GGUF — prefer the re-quantised output if it already exists ──────────
_CANDIDATES = [
    str(IQ4XS_PATH),                    # ✅ already produced today (5.09 GB)
    str(IQ4XS_DIRECT_PATH),             # ✅ direct-requantise variant (5.09 GB)
    r"C:\Users\james\Downloads\gemma4-legal-vlm-q4_k_m.gguf",
    r"C:\Users\james\Downloads\gemma4-e4b-legal-ollama\gemma4-e4b-legal.Q4_K_M.gguf",
    os.path.expandvars(
        r"%USERPROFILE%\.ollama\models\blobs"
        r"\sha256-a79de882a921b9c3781a95a8ef555ea51e7c4dd685a8b2854e9bbe73ab081b43"
    ),
]
SOURCE_GGUF = next((p for p in _CANDIDATES if os.path.exists(p)), None)

# ── LoRA adapter (found locally — no download needed) ─────────────────────────
LORA_DIR = r"C:\Users\james\Downloads\gemma4-legal-final-adapters\gemma4-e4b-legal-grpo-lora"
LORA_OUT = OUT_DIR / "legal-lora-adapter.gguf"
CONVERT_SCRIPT = r"C:\Users\james\Desktop\llama.cpp\convert_lora_to_gguf.py"

# ── Status report ─────────────────────────────────────────────────────────────
print("=" * 60)
print("STATUS — files on disk")
print("=" * 60)

for label, path in [
    ("IQ4_XS (main)",    IQ4XS_PATH),
    ("IQ4_XS (direct)",  IQ4XS_DIRECT_PATH),
    ("Source Q4_K_M",    Q4KM_PATH),
    ("F16 intermediate", F16_PATH),
    ("LoRA adapter",     LORA_OUT),
]:
    p = pathlib.Path(path)
    if p.exists():
        print(f"  ✅ {label:<20} {p.stat().st_size / 1e9:.2f} GB  {p}")
    else:
        print(f"  ❌ {label:<20} not found")

print()
if IQ4XS_PATH.exists():
    print("🎉 Quantisation already complete — no re-run needed.")
    print(f"   Use: ROTORQUANT_MODEL_PATH={IQ4XS_PATH}")
else:
    print("⚠️  IQ4_XS not found — run Steps 1 and 2 below.")


✅ Source GGUF:         C:\Users\james\Downloads\gemma4-legal-vlm-q4_k_m.gguf  (5.3 GB)
✅ llama-quantize:      C:\Users\james\Desktop\llama-server-cuda\llama-quantize.exe
📁 Output directory:   C:\Users\james\Desktop\gemma4-legal-iq4xs

ℹ️  All files found locally — no Colab required.


## Step 1 — Copy source model
We copy the model to a workspace folder with a readable name.

In [13]:
if Q4KM_PATH.exists():
    print(f"⏭  Already copied: {Q4KM_PATH} ({Q4KM_PATH.stat().st_size / 1e9:.1f} GB)")
else:
    print(f"Copying source → {Q4KM_PATH}  (may take a minute)...")
    shutil.copy2(SOURCE_GGUF, Q4KM_PATH)
    print(f"✅ Copied: {Q4KM_PATH.stat().st_size / 1e9:.1f} GB")


⏭  Already copied: C:\Users\james\Desktop\gemma4-legal-iq4xs\gemma4-legal-merged-q4km.gguf (5.3 GB)


## Step 2 — Quantise to IQ4_XS
IQ4_XS uses importance-weighted block-diagonal quantisation — smaller than Q4_K_M with
comparable quality. 

⚠️ We use **`--allow-requantize`** because the source model already has some quantized tensors (like token embeddings).
Takes ~3-5 min. Output is ~4.5 GB.

In [14]:
if IQ4XS_PATH.exists() and IQ4XS_PATH.stat().st_size < 4 * 1024**3:
    print(f"⚠️ Removing incomplete IQ4_XS output: {IQ4XS_PATH} ({IQ4XS_PATH.stat().st_size / 1e9:.1f} GB)")
    IQ4XS_PATH.unlink()
if IQ4XS_PATH.exists():
    print(f"⏭  Already quantised: {IQ4XS_PATH} ({IQ4XS_PATH.stat().st_size / 1e9:.1f} GB)")
else:
    print(f"Quantising to IQ4_XS (direct re-quantisation)...")
    result = subprocess.run(
        [LLAMA_QUANTIZE, "--allow-requantize", str(Q4KM_PATH), str(IQ4XS_PATH), "IQ4_XS"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"llama-quantize failed (exit {result.returncode})")
    print(f"\n✅ IQ4_XS written: {IQ4XS_PATH} ({IQ4XS_PATH.stat().st_size / 1e9:.1f} GB)")


Quantising to IQ4_XS (direct re-quantisation)...

✅ IQ4_XS written: C:\Users\james\Desktop\gemma4-legal-iq4xs\gemma4-legal-iq4xs.gguf (5.1 GB)


## Step 3 — Convert LoRA adapter → GGUF (Path A runtime injection)
Converts `adapter_model.safetensors` to a `.gguf` file that `llama-server` can load at runtime
via `--lora`. The base model **does not** need to be merged first — the adapter is applied on the fly.

- **Input:** `gemma4-e4b-legal-grpo-lora/adapter_model.safetensors` (~162 MB)
- **Output:** `legal-lora-adapter.gguf` (~162 MB)
- **Time:** ~30 seconds
- **Requires:** Python with `torch` and `transformers` installed (uses the llama.cpp convert script)

In [ ]:
if LORA_OUT.exists():
    print(f"⏭  Already converted: {LORA_OUT} ({LORA_OUT.stat().st_size / 1e6:.0f} MB)")
else:
    print("Converting LoRA safetensors → GGUF...")
    print(f"  Input:  {LORA_DIR}")
    print(f"  Output: {LORA_OUT}")
    print()
    result = subprocess.run(
        [
            "python", CONVERT_SCRIPT,
            LORA_DIR,
            "--outfile", str(LORA_OUT),
            "--base-model-id", "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("STDOUT:", result.stdout[-3000:] if result.stdout else "(empty)")
        print("STDERR:", result.stderr[-3000:] if result.stderr else "(empty)")
        raise RuntimeError(f"convert_lora_to_gguf.py failed (exit {result.returncode})")
    print(f"✅ LoRA GGUF: {LORA_OUT} ({LORA_OUT.stat().st_size / 1e6:.0f} MB)")


## Step 3 — Output .env line
Copy the line below into `sveltekit-frontend/.env` then run `npm run turbo:start:rotorquant`.

In [ ]:
print("=" * 60)
print("✅ DONE — paste into sveltekit-frontend/.env")
print("=" * 60)
print()

# Pick best available IQ4_XS
best_iq4xs = next(
    (p for p in [IQ4XS_PATH, IQ4XS_DIRECT_PATH] if p.exists()), None
)

if best_iq4xs:
    print("# Path B — re-quantised merged legal model (LoRA already baked in)")
    print(f"ROTORQUANT_MODEL_PATH={best_iq4xs}")
    print()

if LORA_OUT.exists():
    print("# Path A — runtime LoRA (only use with the HuggingFace BASE IQ4_XS,")
    print("#           NOT with the merged model above — LoRA would be double-applied)")
    print(f"# LEGAL_LORA_PATH={LORA_OUT}")
    print(f"# LEGAL_LORA_SCALE=0.8")
    print()

print("=" * 60)
print()
print("Next steps:")
print("  1. Set ROTORQUANT_MODEL_PATH in sveltekit-frontend/.env")
print("  2. npm run turbo:start:rotorquant")
print("  3. npm run turbo:bench:rotorquant")
print()
print("Optional — delete the 5.34 GB source copy to reclaim space:")
if Q4KM_PATH.exists():
    print(f"  del \"{Q4KM_PATH}\"")
if F16_PATH.exists() and F16_PATH.stat().st_size > 1e9:
    print(f"  del \"{F16_PATH}\"")


Paste this into sveltekit-frontend/.env:

ROTORQUANT_MODEL_PATH=C:\Users\james\Desktop\gemma4-legal-iq4xs\gemma4-legal-iq4xs.gguf


Then run:
  npm run turbo:start:rotorquant
  npm run turbo:bench:rotorquant

